In [ ]:
import numpy as np
from math import erf, sqrt
from pathlib import Path
from astropy.io import fits
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

M = Path("/scratch/s4636708/aida/derived/martini")
SNAP_Z = {17: 5, 21: 4, 25: 3, 33: 2, 50: 1, 67: 0.5}

def gap(line):
    model, snap, sub = line.split()[:3]
    z = SNAP_Z[int(snap)]
    try:
        m1 = np.squeeze(fits.getdata(M / f"z{z:g}" / model / f"gal_{int(sub):06d}" / "bbarolo_3rings/maps/MOCK_1mom.fits"))
    except OSError:
        return None
    row = np.flatnonzero(np.isfinite(m1[int((m1.shape[0] - 1) / 2)])) # pixels along the major axis that have emission
    if not len(row):
        return (z, 999)
    return (z, min(row.min(), m1.shape[1] - 1 - row.max())) # pixels between the major-axis emission and the nearer edge

with ThreadPoolExecutor(8) as pool:
    res = [r for r in pool.map(gap, open(M / "manifest.txt")) if r is not None]

c = Counter(d for z, d in res)
sigma = 60 * 0 + (60**2 - 30**2) ** 0.5 / 2.3548 / 5.0 # BBarolo's blur: Gaussian of FWHM sqrt(60^2 - 30^2) arcsec, in 5" pixels
print(f"{'px from edge':>12} {'galaxies':>9} {'signal lost':>12}")

for d in range(13):
    lost = 0.5 * (1 - erf((d + 0.5) / sigma / sqrt(2))) # share of the blur that falls outside the cube
    print(f"{d:>12} {c.get(d, 0):>9} {100 * lost:>11.1f}%")

for margin in (6, 9, 12):
    cz = Counter(z for z, d in res if d <= margin)
    print(f"within {margin:>2} px: {sum(cz.values())} galaxies  {dict(sorted(cz.items()))}")

px from edge  galaxies  signal lost
           0       241        45.5%
           1       139        36.7%
           2       112        28.6%
           3       108        21.4%
           4        90        15.4%
           5        69        10.6%
           6        59         7.0%
           7        57         4.5%
           8        63         2.7%
           9        65         1.6%
          10        58         0.9%
          11        54         0.5%
          12        56         0.2%
within  6 px: 818 galaxies  {0.5: 331, 1: 275, 2: 143, 3: 61, 4: 8}
within  9 px: 1003 galaxies  {0.5: 390, 1: 341, 2: 179, 3: 81, 4: 12}
within 12 px: 1171 galaxies  {0.5: 435, 1: 386, 2: 234, 3: 99, 4: 17}
